<a href="https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/00_search_engine/notebooks/01_search_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
documents = {
    "S1": "The best homemade pizza recipe for beginners",
    "S2": "How to make the best pizza dough from scratch recipe",
    "S3": "Top 10 tips for baking pizza at home for beginners",
    "S4": "Easy pasta recipes and Italian cooking techniques",
    "S5": "A guide to building your own outdoor pizza oven in summer",
    "S6": "The history of pizza from Naples to New York for foodies",
    "S7": "Healthy meal prep ideas and cooking tips for temperature control",
    "S8": "Best wood-fired pizza restaurants in the best cities",
    "S9": "How to choose the right flour for homemade bread and pizza",
    "S10": "Italian cheese varieties and their uses in traditional cooking"
}

In [3]:
def tokenise(text):
    return text.lower().split()

for doc_id, text in documents.items():
    tokens = tokenise(text)
    print(f"{doc_id}: {len(tokens)} tokens → {tokens[:5]}...")

S1: 7 tokens → ['the', 'best', 'homemade', 'pizza', 'recipe']...
S2: 10 tokens → ['how', 'to', 'make', 'the', 'best']...
S3: 10 tokens → ['top', '10', 'tips', 'for', 'baking']...
S4: 7 tokens → ['easy', 'pasta', 'recipes', 'and', 'italian']...
S5: 11 tokens → ['a', 'guide', 'to', 'building', 'your']...
S6: 11 tokens → ['the', 'history', 'of', 'pizza', 'from']...
S7: 10 tokens → ['healthy', 'meal', 'prep', 'ideas', 'and']...
S8: 8 tokens → ['best', 'wood-fired', 'pizza', 'restaurants', 'in']...
S9: 11 tokens → ['how', 'to', 'choose', 'the', 'right']...
S10: 9 tokens → ['italian', 'cheese', 'varieties', 'and', 'their']...


In [4]:
STOP_WORDS = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at',
              'to', 'for', 'of', 'is', 'it', 'from', 'how', 'your',
              'their', 'own', 'that', 'this'}

def remove_stop_words(tokens):
    return [t for t in tokens if t not in STOP_WORDS]

# Example with S1
tokens = tokenise(documents["S1"])
filtered = remove_stop_words(tokens)
print(f"Before: {tokens}")
print(f"After:  {filtered}")
print(f"Removed: {len(tokens) - len(filtered)} / {len(tokens)} "
      f"({(len(tokens)-len(filtered))/len(tokens)*100:.0f}% were stop words)")

Before: ['the', 'best', 'homemade', 'pizza', 'recipe', 'for', 'beginners']
After:  ['best', 'homemade', 'pizza', 'recipe', 'beginners']
Removed: 2 / 7 (29% were stop words)


In [5]:
STEM_RULES = {
    'recipes': 'recip', 'recipe': 'recip', 'baking': 'bake',
    'cooking': 'cook', 'homemade': 'homemad', 'beginners': 'beginn',
    'building': 'build', 'cities': 'citi', 'varieties': 'varieti',
    'ideas': 'idea', 'techniques': 'techniqu', 'traditional': 'tradit',
    'restaurants': 'restaur', 'healthy': 'healthi', 'uses': 'use',
    'tips': 'tip', 'choose': 'choos', 'fired': 'fire',
}

def stem(token):
    return STEM_RULES.get(token, token)

# Example
words = ['recipes', 'cooking', 'baking', 'beginners', 'cities']
for w in words:
    print(f"  {w:15s} → {stem(w)}")

  recipes         → recip
  cooking         → cook
  baking          → bake
  beginners       → beginn
  cities          → citi


In [6]:
def process(text):
    tokens = tokenise(text)
    filtered = remove_stop_words(tokens)
    stemmed = [stem(t) for t in filtered]
    return stemmed

processed = {}
for doc_id, text in documents.items():
    processed[doc_id] = process(text)
    print(f"{doc_id}: {processed[doc_id]}")

S1: ['best', 'homemad', 'pizza', 'recip', 'beginn']
S2: ['make', 'best', 'pizza', 'dough', 'scratch', 'recip']
S3: ['top', '10', 'tip', 'bake', 'pizza', 'home', 'beginn']
S4: ['easy', 'pasta', 'recip', 'italian', 'cook', 'techniqu']
S5: ['guide', 'build', 'outdoor', 'pizza', 'oven', 'summer']
S6: ['history', 'pizza', 'naples', 'new', 'york', 'foodies']
S7: ['healthi', 'meal', 'prep', 'idea', 'cook', 'tip', 'temperature', 'control']
S8: ['best', 'wood-fired', 'pizza', 'restaur', 'best', 'citi']
S9: ['choos', 'right', 'flour', 'homemad', 'bread', 'pizza']
S10: ['italian', 'cheese', 'varieti', 'use', 'tradit', 'cook']


In [7]:
from collections import defaultdict

inverted_index = defaultdict(set)
for doc_id, terms in processed.items():
    for term in terms:
        inverted_index[term].add(doc_id)

# Show a few entries
for term in ['pizza', 'recip', 'cook', 'best', 'beginn']:
    docs = sorted(inverted_index[term])
    print(f"  {term:10s} → {docs}")

  pizza      → ['S1', 'S2', 'S3', 'S5', 'S6', 'S8', 'S9']
  recip      → ['S1', 'S2', 'S4']
  cook       → ['S10', 'S4', 'S7']
  best       → ['S1', 'S2', 'S8']
  beginn     → ['S1', 'S3']


In [8]:
import math

def tf(term, doc_terms):
    """Term Frequency: count / total terms in doc"""
    return doc_terms.count(term) / len(doc_terms)

def idf(term, all_docs):
    """Inverse Document Frequency: log(N / docs containing term)"""
    n = len(all_docs)
    df = sum(1 for doc in all_docs.values() if term in doc)
    return math.log(n / df) if df > 0 else 0

def tf_idf(term, doc_terms, all_docs):
    return tf(term, doc_terms) * idf(term, all_docs)

# Score all docs for query "best pizza recipe"
query = ["best", "pizza", "recip"]

print(f"{'Doc':<5} {'best':>8} {'pizza':>8} {'recip':>8} {'TOTAL':>8}")
print("-" * 42)

scores = {}
for doc_id, terms in processed.items():
    doc_scores = {q: tf_idf(q, terms, processed) for q in query}
    total = sum(doc_scores.values())
    scores[doc_id] = total
    if total > 0:
        print(f"{doc_id:<5} {doc_scores['best']:8.4f} "
              f"{doc_scores['pizza']:8.4f} "
              f"{doc_scores['recip']:8.4f} {total:8.4f}")

# Rank results
print("\n🏆 Ranked Results:")
for rank, (doc_id, score) in enumerate(
    sorted(scores.items(), key=lambda x: -x[1])[:5], 1
):
    print(f"  {rank}. {doc_id} (score: {score:.4f}) — {documents[doc_id]}")

Doc       best    pizza    recip    TOTAL
------------------------------------------
S1      0.2408   0.0713   0.2408   0.5529
S2      0.2007   0.0594   0.2007   0.4608
S3      0.0000   0.0510   0.0000   0.0510
S4      0.0000   0.0000   0.2007   0.2007
S5      0.0000   0.0594   0.0000   0.0594
S6      0.0000   0.0594   0.0000   0.0594
S8      0.4013   0.0594   0.0000   0.4608
S9      0.0000   0.0594   0.0000   0.0594

🏆 Ranked Results:
  1. S1 (score: 0.5529) — The best homemade pizza recipe for beginners
  2. S2 (score: 0.4608) — How to make the best pizza dough from scratch recipe
  3. S8 (score: 0.4608) — Best wood-fired pizza restaurants in the best cities
  4. S4 (score: 0.2007) — Easy pasta recipes and Italian cooking techniques
  5. S5 (score: 0.0594) — A guide to building your own outdoor pizza oven in summer


In [9]:
def search(query_text, documents, processed, top_k=5):
    # Step 1: Process query through same pipeline
    query_terms = process(query_text)
    print(f"Query: '{query_text}' → {query_terms}")

    # Step 2: Find candidate docs via inverted index
    candidates = set()
    for term in query_terms:
        candidates |= inverted_index.get(term, set())
    print(f"Candidates: {sorted(candidates)}")

    # Step 3: Score with TF-IDF
    scores = {}
    for doc_id in candidates:
        score = sum(tf_idf(t, processed[doc_id], processed)
                    for t in query_terms)
        scores[doc_id] = score

    # Step 4: Rank
    ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_k]

    print(f"\n📋 Top {top_k} Results:")
    for rank, (doc_id, score) in enumerate(ranked, 1):
        print(f"  {rank}. [{score:.4f}] {documents[doc_id]}")

    return ranked

# Try it!
search("best pizza recipe", documents, processed)

Query: 'best pizza recipe' → ['best', 'pizza', 'recip']
Candidates: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S8', 'S9']

📋 Top 5 Results:
  1. [0.5529] The best homemade pizza recipe for beginners
  2. [0.4608] Best wood-fired pizza restaurants in the best cities
  3. [0.4608] How to make the best pizza dough from scratch recipe
  4. [0.2007] Easy pasta recipes and Italian cooking techniques
  5. [0.0594] The history of pizza from Naples to New York for foodies


[('S1', 0.552924110518121),
 ('S8', 0.4607700920984341),
 ('S2', 0.4607700920984341),
 ('S4', 0.20066213405432268),
 ('S6', 0.05944582398978873)]

In [11]:
search("Italian traditional cooking", documents, processed)

Query: 'Italian traditional cooking' → ['italian', 'tradit', 'cook']
Candidates: ['S10', 'S4', 'S7']

📋 Top 5 Results:
  1. [0.8527] Italian cheese varieties and their uses in traditional cooking
  2. [0.4689] Easy pasta recipes and Italian cooking techniques
  3. [0.1505] Healthy meal prep ideas and cooking tips for temperature control


[('S10', 0.852665968292347),
 ('S4', 0.46890178612667266),
 ('S7', 0.15049660054074201)]